# Семинар 2a. Особые точки и гомография

На втором семинаре мы уже склеили панораму готовыми SIFT и `findHomography`. Сегодня разбираем этот конвейер по частям и меряем каждую часть. Пишем детектор Харриса руками, проверяем, какие детекторы находят те же точки после поворота, масштаба, размытия и смены яркости. Сравниваем бинарные и вещественные дескрипторы и разные способы отсева пар. Пишем гомографию по четырём точкам и RANSAC сами и сравниваем с OpenCV. Всё это на паре снимков граффити с известной истинной гомографией, а в конце ищем коробку на захламлённом столе.

Нужны `opencv-python` версии 4.4 или новее (там есть SIFT), `numpy`, `matplotlib`. Картинки из примеров OpenCV (около 2 МБ) скачиваются в `data/opencv`.

In [ ]:
import os
import time
import urllib.request

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(suppress=True, precision=4)
os.makedirs("data/opencv", exist_ok=True)
URL = "https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/"
for n in ["graf1.png", "graf3.png", "H1to3p.xml", "box.png", "box_in_scene.png"]:
    if not os.path.exists("data/opencv/" + n):
        urllib.request.urlretrieve(URL + n, "data/opencv/" + n)
print(cv.__version__)

## Данные с известным ответом

Два снимка одной стены с граффити под разными углами из набора Oxford Affine. Стена плоская, поэтому снимки связаны гомографией, и авторы её измерили. Перенесём первый снимок гомографией на второй и сравним.

In [ ]:
g1 = cv.imread("data/opencv/graf1.png", cv.IMREAD_GRAYSCALE)
g3 = cv.imread("data/opencv/graf3.png", cv.IMREAD_GRAYSCALE)
fs = cv.FileStorage("data/opencv/H1to3p.xml", cv.FILE_STORAGE_READ)
H13 = fs.getNode("H13").mat()
print(g1.shape, g3.shape)
print("H13 =\n", H13)
warped = cv.warpPerspective(g1, H13, (g3.shape[1], g3.shape[0]))
valid = cv.warpPerspective(np.ones_like(g1), H13, (g3.shape[1], g3.shape[0])) > 0
fig, axes = plt.subplots(1, 4, figsize=(18, 3.8))
for ax, im, t in zip(axes, [g1, g3, warped, np.abs(warped.astype(int) - g3) * valid], ["graf1", "graf3", "graf1, перенесённый H13", "разница с graf3"]):
    ax.imshow(im, cmap="gray"); ax.set_title(t); ax.axis("off")
plt.show()
print("средняя разница яркости в перекрытии: %.1f из 255" % np.abs(warped.astype(int) - g3)[valid].mean())

def apply_h(Hm, pts):
    p = np.c_[pts, np.ones(len(pts))] @ Hm.T
    with np.errstate(divide="ignore", invalid="ignore"):
        return p[:, :2] / p[:, 2:]

**Что видно:** снимки 800 на 640, второй снимок сделан с поворотом камеры примерно на 30 градусов. Гомография не просто поворот: в последней строке есть ненулевой элемент 3.5e−4, это перспектива. Первый снимок, перенесённый H13, почти совпадает со вторым: средняя разница яркости 16 из 255, и в карте разницы видны только тонкие контуры. Остаток даёт другое освещение и размытие при пересчёте пикселей. Значит, H13 можно считать правдой и проверять ею всё остальное.

## Харрис руками

Для каждого пикселя считаем структурный тензор: суммы Ix², IxIy, Iy² в гауссовом окне. Его собственные числа говорят, в скольких направлениях меняется яркость. Отклик Харриса R = det M − k (tr M)² большой, когда оба собственных числа большие. Сравниваем с `cv.cornerHarris`.

In [ ]:
def harris(img, sigma=1.5, k=0.04):
    f = img.astype(np.float32) / 255
    Ix = cv.Sobel(f, cv.CV_32F, 1, 0, ksize=3)
    Iy = cv.Sobel(f, cv.CV_32F, 0, 1, ksize=3)
    Sxx = cv.GaussianBlur(Ix * Ix, (0, 0), sigma)
    Syy = cv.GaussianBlur(Iy * Iy, (0, 0), sigma)
    Sxy = cv.GaussianBlur(Ix * Iy, (0, 0), sigma)
    det, tr = Sxx * Syy - Sxy ** 2, Sxx + Syy
    return det - k * tr ** 2, (Sxx, Syy, Sxy)

def local_max(R, n=500, r=5):
    dil = cv.dilate(R, np.ones((2 * r + 1, 2 * r + 1)))
    ys, xs = np.nonzero((R == dil) & (R > 0))
    order = np.argsort(-R[ys, xs])[:n]
    return np.c_[xs[order], ys[order]].astype(np.float32)

R, (Sxx, Syy, Sxy) = harris(g1)
Rcv = cv.cornerHarris(g1.astype(np.float32) / 255, 3, 3, 0.04)
print("корреляция нашего отклика с cv.cornerHarris: %.3f" % np.corrcoef(R.ravel(), Rcv.ravel())[0, 1])
pts = local_max(R)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].imshow(np.sign(R) * np.abs(R) ** 0.5, cmap="RdBu_r", vmin=-0.05, vmax=0.05); axes[0].set_title("отклик Харриса: красное углы, синее края")
axes[1].imshow(g1, cmap="gray"); axes[1].plot(pts[:, 0], pts[:, 1], "r+", ms=5); axes[1].set_title("500 сильнейших локальных максимумов")
for a in axes:
    a.axis("off")
plt.show()

**Что видно:** наш отклик совпадает с `cv.cornerHarris` с корреляцией 0.81: формула та же, отличаются размер окна (у нас гауссово с sigma 1.5, у OpenCV квадрат 3 на 3) и масштаб производных. Отрицательный отклик (синий) идёт вдоль краёв, положительный (красный) пятнами на углах, пересечениях линий и мелких деталях. 500 сильнейших максимумов сидят на глазах, пальцах, буквах, а на длинных плавных линиях их нет.

In [ ]:
l1 = (Sxx + Syy) / 2 + np.sqrt(((Sxx - Syy) / 2) ** 2 + Sxy ** 2)
l2 = (Sxx + Syy) / 2 - np.sqrt(((Sxx - Syy) / 2) ** 2 + Sxy ** 2)
sel = rng.choice(l1.size, 20000, replace=False)
fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(l2.ravel()[sel], l1.ravel()[sel], s=2, c=np.clip(R.ravel()[sel], -0.01, 0.01), cmap="RdBu_r")
ax.set_xscale("symlog", linthresh=1e-4); ax.set_yscale("symlog", linthresh=1e-4)
ax.set_xlabel("меньшее собственное число"); ax.set_ylabel("большее собственное число")
plt.show()
t = 0.01 * l1.max()
print("порог %.4f, 1%% от максимума; доля пикселей: ровные %.2f, края %.2f, углы %.3f" % (t, np.mean(l1 < t), np.mean((l1 >= t) & (l2 < t)), np.mean(l2 >= t)))

**Что видно:** на графике собственных чисел три области. Внизу слева ровные места: оба числа маленькие. Вдоль верхнего края края: одно число большое, другое на 2 до 3 порядка меньше. Справа вдоль диагонали углы: оба числа большие, и отклик Харриса там положительный. При пороге в 1% от максимума половина пикселей ровные, 41% лежат на краях и только 6% это углы. Углов мало, и поэтому они ценны: по ним можно однозначно найти место.

## Повторяемость детекторов

Хорошая особая точка находится снова, когда снимок поменялся. Берём graf1, применяем к нему известное преобразование и считаем, какая доля точек, найденных на исходнике, после переноса точной гомографией имеет найденную точку на новом снимке в пределах 3 пикселей. Детекторы: Харрис (`goodFeaturesToTrack`), FAST, ORB и SIFT, у каждого берём 500 точек.

In [ ]:
def detect(name, img, n=500):
    if name == "Харрис":
        p = cv.goodFeaturesToTrack(img, n, 0.001, 5, useHarrisDetector=True)
        return p[:, 0]
    det = {"FAST": cv.FastFeatureDetector_create(20), "ORB": cv.ORB_create(n), "SIFT": cv.SIFT_create(n)}[name]
    kps = sorted(det.detect(img, None), key=lambda k: -k.response)[:n]
    return np.float32([k.pt for k in kps])

def rot_scale_h(img, angle, scale):
    h, w = img.shape
    A = cv.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    return np.vstack([A, [0, 0, 1]])

def repeatability(name, img, Hm, img2, tol=3.0):
    a, b = detect(name, img), detect(name, img2)
    pa = apply_h(Hm, a)
    inside = (pa[:, 0] >= 0) & (pa[:, 0] < img2.shape[1]) & (pa[:, 1] >= 0) & (pa[:, 1] < img2.shape[0])
    pa = pa[inside]
    d = np.sqrt(((pa[:, None] - b[None]) ** 2).sum(2)).min(1)
    return (d < tol).mean()

tests = {}
for ang in [0, 15, 45, 90]:
    Hm = rot_scale_h(g1, ang, 1.0)
    tests[f"поворот {ang}°"] = (Hm, cv.warpPerspective(g1, Hm, g1.shape[::-1]))
for sc in [0.7, 0.5]:
    Hm = rot_scale_h(g1, 0, sc)
    tests[f"масштаб {sc}"] = (Hm, cv.warpPerspective(g1, Hm, g1.shape[::-1]))
tests["размытие σ=2"] = (np.eye(3), cv.GaussianBlur(g1, (0, 0), 2))
tests["яркость ×0.5"] = (np.eye(3), (g1 * 0.5).astype(np.uint8))
tests["graf1 в graf3"] = (H13, g3)

names = ["Харрис", "FAST", "ORB", "SIFT"]
table = np.array([[repeatability(n, g1, Hm, im2) for n in names] for Hm, im2 in tests.values()])
print(f"{'':16s}" + "".join(f"{n:>9s}" for n in names))
for (t, _), row in zip(tests.items(), table):
    print(f"{t:16s}" + "".join(f"{v:9.2f}" for v in row))

**Что видно:** при повороте все детекторы находят снова 80 до 100% точек: Харрис и FAST основаны на структурном тензоре и круге пикселей, им поворот не важен. При уменьшении вдвое Харрис и FAST теряют четверть точек, ORB держится лучше (0.84), потому что ищет точки на пирамиде масштабов. Размытие сильнее всего бьёт по FAST (0.42) и SIFT (0.48): мелкие детали пропадают. Яркость ×0.5 почти ничего не меняет. На настоящей паре graf1 и graf3 с поворотом стены и перспективой повторяемость падает до 0.45 до 0.73. Неожиданно SIFT тут не лучший: при 500 точках на кадр он находит много мелких точек, которые на втором снимке под углом не находятся. Повторяемость зависит от того, сколько точек мы берём и насколько сильно меняется вид.

## Дескрипторы

ORB описывает окрестность 256 битами: результатами сравнений яркостей пар пикселей. Расстояние Хэмминга считает несовпавшие биты. SIFT описывает окрестность 128 числами: гистограммами направлений градиента в сетке 4 на 4. Расстояние евклидово. Посмотрим на размеры, скорость и на то, как далеко правильная пара от неправильных.

In [ ]:
orb, sift = cv.ORB_create(2000), cv.SIFT_create(2000)
feats = {}
for name, det in [("ORB", orb), ("SIFT", sift)]:
    t0 = time.time()
    k1, d1 = det.detectAndCompute(g1, None)
    k2, d2 = det.detectAndCompute(g3, None)
    feats[name] = (k1, d1, k2, d2)
    print(f"{name}: точек {len(k1)} и {len(k2)}, дескриптор {d1.shape[1]} x {d1.dtype}, {d1.nbytes // len(k1)} байт, {time.time() - t0:.2f} с на два снимка")

a = np.array([0b10110000], np.uint8); b = np.array([0b01110000], np.uint8)
print("байты 176 и 112: разница чисел", 176 - 112, ", расстояние Хэмминга", cv.norm(a, b, cv.NORM_HAMMING))

In [ ]:
def true_pairs(k1, k2, tol=3.0):
    p1 = np.float32([k.pt for k in k1]); p2 = np.float32([k.pt for k in k2])
    pr = apply_h(H13, p1)
    d = np.sqrt(((pr[:, None] - p2[None]) ** 2).sum(2))
    return d < tol

fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
for ax, (name, norm) in zip(axes, [("ORB", cv.NORM_HAMMING), ("SIFT", cv.NORM_L2)]):
    k1, d1, k2, d2 = feats[name]
    ok = true_pairs(k1, k2)
    if name == "ORB":
        b1, b2 = np.unpackbits(d1, axis=1).astype(np.float32), np.unpackbits(d2, axis=1).astype(np.float32)
        D = b1 @ (1 - b2).T + (1 - b1) @ b2.T
    else:
        a1, a2 = d1.astype(np.float64), d2.astype(np.float64)
        D = np.sqrt(np.maximum((a1 ** 2).sum(1)[:, None] + (a2 ** 2).sum(1)[None] - 2 * a1 @ a2.T, 0))
    has = ok.any(1)
    pos = D[ok]
    neg = D[has][~ok[has]]
    ax.hist(neg, 60, density=True, alpha=0.6, label="неправильные пары")
    ax.hist(pos, 60, density=True, alpha=0.6, label="правильные пары")
    ax.set_title(name); ax.legend()
    print(f"{name}: точек, у которых есть настоящая пара: {has.sum()} из {len(k1)}")
plt.show()

**Что видно:** ORB хранит 32 байта на точку и считается за 0.03 с на два снимка, SIFT 512 байт и 0.18 с. Байты 176 и 112 различаются на 64 как числа, но всего в 2 битах, поэтому расстояние между бинарными дескрипторами считают по Хэммингу. На гистограммах правильные пары в среднем ближе неправильных, но распределения сильно перекрываются: у ORB правильные пары от 30 до 170 бит, неправильные от 80 до 200. Одним порогом на расстояние их не разделить, поэтому смотрят на соседей.

## Отбор пар

Для каждой точки первого снимка ищем ближайший дескриптор на втором. Три правила отбора: брать всех ближайших; тест отношения Лоу (ближайший должен быть заметно ближе второго); взаимная проверка (точки должны быть ближайшими друг для друга). Правильность пары проверяем истинной гомографией с допуском 3 пикселя.

In [ ]:
def match_quality(name, rule, ratio=0.8):
    k1, d1, k2, d2 = feats[name]
    norm = cv.NORM_HAMMING if name == "ORB" else cv.NORM_L2
    if rule == "взаимная":
        ms = cv.BFMatcher(norm, crossCheck=True).match(d1, d2)
    else:
        knn = cv.BFMatcher(norm).knnMatch(d1, d2, k=2)
        ms = [m for m, n in knn if rule == "все" or m.distance < ratio * n.distance]
    p1 = np.float32([k1[m.queryIdx].pt for m in ms]); p2 = np.float32([k2[m.trainIdx].pt for m in ms])
    good = np.linalg.norm(apply_h(H13, p1) - p2, axis=1) < 3
    return len(ms), good.sum(), good.mean() if len(ms) else 0, p1, p2, good

for name in ["ORB", "SIFT"]:
    for rule in ["все", "Лоу 0.8", "взаимная"]:
        n, g, prec, *_ = match_quality(name, rule)
        print(f"{name:4s} {rule:9s}: пар {n:5d}, правильных {g:4d}, точность {prec:.2f}")

**Что видно:** если брать для каждой точки ближайшего соседа, только 30% пар правильные у ORB и 22% у SIFT. Тест Лоу с порогом 0.8 оставляет 288 пар у ORB, из них 66% правильные, у SIFT 527 пар и 56%. Взаимная проверка оставляет больше правильных пар (330 и 392), но с точностью около 47%. На этой паре снимков ORB даже чуть лучше SIFT: перспективу 30° плохо переносят оба, а ORB находит больше точек на крупных деталях.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
for name in ["ORB", "SIFT"]:
    rows = [match_quality(name, "Лоу", r)[:3] for r in np.arange(0.5, 1.01, 0.05)]
    rows = np.array(rows)
    ax.plot(rows[:, 1], rows[:, 2], "o-", label=name)
    for k, (r, (n, g, p)) in enumerate(zip(np.arange(0.5, 1.01, 0.05)[::2], rows[::2])):
        off = (8, 8) if name == "ORB" else (-24, -14)
        if name == "ORB" and k == 0:
            off = (-22, 4)
        ax.annotate(f"{r:.1f}", (g, p), xytext=off, textcoords="offset points", fontsize=8, color=ax.lines[-1].get_color())
ax.set_xlabel("правильных пар"); ax.set_ylabel("доля правильных"); ax.legend(); ax.set_title("тест Лоу при разном пороге"); ax.set_ylim(0.1, 1.0)
plt.show()

**Что видно:** порог теста Лоу двигает по кривой: строгий порог 0.5 даёт 80 до 90% правильных пар, но их всего 10 до 20, мягкий 1.0 даёт все пары с точностью 22 до 30%. Для RANSAC важны оба числа: правильных пар должно быть хотя бы несколько десятков, а их доля определяет, сколько попыток понадобится. Порог 0.75 до 0.8 из статьи Лоу хороший компромисс и здесь.

## Гомография по четырём точкам

Каждая пара точек даёт два линейных уравнения на 9 элементов H, H определена с точностью до множителя, значит хватает четырёх пар. Собираем матрицу 8 на 9 и берём вектор нулевого пространства через SVD. Перед этим нормируем точки (сдвиг в центр масс и масштаб): без нормировки уравнения плохо обусловлены. Сравниваем с `cv.getPerspectiveTransform` и с истинной H.

In [ ]:
def normalize(p):
    c = p.mean(0); s = np.sqrt(2) / np.linalg.norm(p - c, axis=1).mean()
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])
    return apply_h(T, p), T

def dlt(p1, p2, norm=True):
    if norm:
        p1, T1 = normalize(p1); p2, T2 = normalize(p2)
    A = []
    for (x, y), (u, v) in zip(p1, p2):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    Hm = np.linalg.svd(np.array(A))[2][-1].reshape(3, 3)
    if norm:
        Hm = np.linalg.inv(T2) @ Hm @ T1
    return Hm / Hm[2, 2]

src = np.float32([[100, 100], [700, 120], [680, 560], [80, 520]])
dst = apply_h(H13, src).astype(np.float32)
H4 = dlt(src, dst)
print("наш DLT по 4 точкам:\n", H4)
print("разница с getPerspectiveTransform:", np.abs(H4 - cv.getPerspectiveTransform(src, dst)).max())
print("разница с истинной H13:", np.abs(H4 - H13 / H13[2, 2]).max())
noisy = dst + rng.normal(0, 1, dst.shape)
grid = np.float32([[x, y] for x in range(0, 800, 50) for y in range(0, 640, 50)])
for norm in [True, False]:
    errs = []
    for rep in range(200):
        pts_ = np.float32(rng.uniform([0, 0], [800, 640], (12, 2)))
        dd = apply_h(H13, pts_) + rng.normal(0, 1, (12, 2))
        errs.append(np.linalg.norm(apply_h(dlt(pts_, dd, norm), grid) - apply_h(H13, grid), axis=1).mean())
    print(f"12 точек с шумом 1 px, {'с нормировкой' if norm else 'без нормировки'}: средняя ошибка переноса {np.median(errs):.2f} px")

def dlt_matrix(p1, p2):
    return np.array([r for (x, y), (u, v) in zip(p1, p2) for r in
                     ([-x, -y, -1, 0, 0, 0, u * x, u * y, u], [0, 0, 0, -x, -y, -1, v * x, v * y, v])])
q1, _ = normalize(src); q2, _ = normalize(dst)
print("число обусловленности матрицы уравнений: без нормировки %.1e, с нормировкой %.1e" % (np.linalg.cond(dlt_matrix(src, dst)), np.linalg.cond(dlt_matrix(q1, q2))))

**Что видно:** по четырём точным парам наш DLT восстанавливает H13 до шестого знака и совпадает с `getPerspectiveTransform`. С 12 зашумлёнными парами ошибка переноса около 0.9 пикселя и с нормировкой, и без неё: в float64 при координатах до 800 разницы не видно. Но число обусловленности системы без нормировки 1.5e6, с нормировкой 3.4: без нормировки малые ошибки в данных могут усиливаться в миллион раз, и в float32 или с большими координатами это уже заметно.

## RANSAC руками

Пары после теста Лоу всё равно содержат ошибки. RANSAC: много раз берём 4 случайные пары, строим по ним H, считаем, сколько всех пар с ней согласны в пределах порога. Лучшую гипотезу уточняем по всем согласным парам. Число попыток, чтобы с вероятностью p хоть раз взять 4 правильные пары при доле правильных w: N = log(1 − p) / log(1 − w⁴).

In [ ]:
def ransac_h(p1, p2, thr=3.0, iters=2000, seed=0):
    g = np.random.default_rng(seed)
    best, best_in = None, np.zeros(len(p1), bool)
    history = []
    for it in range(iters):
        idx = g.choice(len(p1), 4, replace=False)
        try:
            Hm = dlt(p1[idx], p2[idx])
        except np.linalg.LinAlgError:
            continue
        err = np.linalg.norm(apply_h(Hm, p1) - p2, axis=1)
        inl = err < thr
        if inl.sum() > best_in.sum():
            best, best_in = Hm, inl
        history.append(best_in.sum())
    return dlt(p1[best_in], p2[best_in]), best_in, history

n, g, prec, p1, p2, good = match_quality("ORB", "Лоу 0.8")
t0 = time.time()
H_r, inl, hist = ransac_h(p1, p2)
print(f"пар {n}, из них правильных {good.mean():.2f}; RANSAC за {time.time() - t0:.1f} с: inliers {inl.sum()}, из них правильных {good[inl].mean():.3f}")
H_cv, m_cv = cv.findHomography(p1, p2, cv.RANSAC, 3.0)
corners = np.float32([[0, 0], [800, 0], [800, 640], [0, 640]])
for name, Hm in [("наш RANSAC", H_r), ("cv.findHomography", H_cv), ("МНК по всем парам", dlt(p1, p2))]:
    e = np.linalg.norm(apply_h(Hm, corners) - apply_h(H13, corners), axis=1)
    print(f"{name:18s}: ошибка переноса углов снимка {e.mean():8.2f} px")
for w in [0.2, 0.5, 0.8]:
    print(f"доля правильных {w}: нужно {int(np.ceil(np.log(1 - 0.99) / np.log(1 - w ** 4)))} попыток для 99%")
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(hist); ax.set_xscale("log"); ax.set_xlabel("попытка"); ax.set_ylabel("лучшее число inliers")
plt.show()

**Что видно:** из 288 пар после теста Лоу правильных 66%. Наш RANSAC нашёл 187 inliers, и все они правильные. Ошибка переноса углов снимка 1.1 пикселя, у `cv.findHomography` 1.3. Метод наименьших квадратов по всем парам ошибается на 166 пикселей: треть неправильных пар тянет решение куда угодно. По графику лучшая гипотеза нашлась примерно за 140 попыток, дальше 2000 попыток ничего не добавили. Формула для 66% правильных требует около 20 попыток для 99%, и на практике лучшая гипотеза находится быстро, а остаток уходит на проверку.

## Порог RANSAC и доля выбросов

Синтетический опыт: 100 правильных пар с шумом 1 пиксель и разная доля случайных пар. Сравниваем ошибку переноса углов у МНК по всем парам и у RANSAC с разным порогом.

In [ ]:
for out_frac in [0.1, 0.3, 0.5, 0.7]:
    row = []
    for thr in [1, 3, 10]:
        es, ls = [], []
        for rep in range(10):
            g_ = np.random.default_rng(rep)
            n_out = int(100 * out_frac / (1 - out_frac))
            a_ = g_.uniform([0, 0], [800, 640], (100 + n_out, 2))
            b_ = apply_h(H13, a_) + g_.normal(0, 1, a_.shape)
            b_[100:] = g_.uniform([0, 0], [800, 640], (n_out, 2))
            Hm, _, _ = ransac_h(a_, b_, thr=thr, iters=500, seed=rep)
            es.append(np.linalg.norm(apply_h(Hm, corners) - apply_h(H13, corners), axis=1).mean())
            ls.append(np.linalg.norm(apply_h(dlt(a_, b_), corners) - apply_h(H13, corners), axis=1).mean())
        row.append(np.median(es))
    print(f"выбросов {out_frac:.0%}: МНК {np.median(ls):8.1f} px; RANSAC порог 1: {row[0]:5.2f}, порог 3: {row[1]:5.2f}, порог 10: {row[2]:5.2f}")

**Что видно:** МНК по всем парам ломается уже при 10% выбросов: 90 пикселей ошибки. RANSAC держится до 50% выбросов при любом пороге. При 70% выбросов и 500 попытках строгий порог в 1 пиксель уже не справляется (10 пикселей): при шуме 1 пиксель часть правильных пар не попадает в порог, и хорошая гипотеза набирает мало голосов. Порог 3 пикселя, примерно 3 сигмы шума, хороший выбор, а слишком большой порог 10 пикселей здесь не мешает, потому что случайные пары разбросаны по всему кадру.

## Результат на граффити

Пары, которые RANSAC признал правильными, и выпрямление второго снимка в геометрию первого.

In [ ]:
k1, d1, k2, d2 = feats["ORB"]
img_m = np.hstack([cv.cvtColor(g1, cv.COLOR_GRAY2RGB), cv.cvtColor(g3, cv.COLOR_GRAY2RGB)])
fig, axes = plt.subplots(1, 2, figsize=(18, 5.5), gridspec_kw=dict(width_ratios=[2, 1]))
axes[0].imshow(img_m)
sel = np.nonzero(inl)[0][::4]
for i in sel:
    axes[0].plot([p1[i, 0], p2[i, 0] + 800], [p1[i, 1], p2[i, 1]], "-", lw=0.6, color="lime")
out_idx = np.nonzero(~inl)[0][:40]
for i in out_idx:
    axes[0].plot([p1[i, 0], p2[i, 0] + 800], [p1[i, 1], p2[i, 1]], "-", lw=0.6, color="red")
axes[0].set_title("зелёные: inliers (каждая четвёртая), красные: отброшенные"); axes[0].axis("off")
back = cv.warpPerspective(g3, np.linalg.inv(H_r), (800, 640))
axes[1].imshow(np.dstack([g1, back, g1])); axes[1].set_title("graf1 и выпрямленный graf3: серое совпало"); axes[1].axis("off")
plt.show()

**Что видно:** зелёные пары идут почти параллельно и согласованно, красные пересекают картинку как попало. Если выпрямить второй снимок найденной гомографией и наложить на первый (первый в красном и синем каналах, второй в зелёном), совпавшие места выглядят серыми, а цветные края остаются только там, где разное освещение и вне стены.

## Второй набор: коробка на столе

Найдём коробку на снимке стола с другими предметами. Здесь плоская только сама коробка, вокруг другие предметы, и правильных пар гораздо меньше.

Сравнивать детекторы честно можно двумя способами: при одинаковом числе точек или при одинаковом времени. Сначала даём обоим не больше 500 точек на снимок. Потом ORB с 2000 точек: даже так он работает быстрее, чем SIFT с 500.

In [ ]:
box = cv.imread("data/opencv/box.png", cv.IMREAD_GRAYSCALE)
scene = cv.imread("data/opencv/box_in_scene.png", cv.IMREAD_GRAYSCALE)
print(box.shape, scene.shape)

def find_box(name, n, reps=5):
    det = cv.ORB_create(n) if name == "ORB" else cv.SIFT_create(n)
    norm = cv.NORM_HAMMING if name == "ORB" else cv.NORM_L2
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        ka, da = det.detectAndCompute(box, None)
        kb, db = det.detectAndCompute(scene, None)
        knn = cv.BFMatcher(norm).knnMatch(da, db, k=2)
        times.append(time.perf_counter() - t0)
    ms = [m for m, n_ in knn if m.distance < 0.8 * n_.distance]
    pa = np.float32([ka[m.queryIdx].pt for m in ms]); pb = np.float32([kb[m.trainIdx].pt for m in ms])
    Hb, mb = cv.findHomography(pa, pb, cv.RANSAC, 4.0)
    return ka, kb, ms, pb, Hb, mb.ravel() > 0, np.median(times)

configs = [("ORB", 500), ("SIFT", 500), ("ORB", 2000)]
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (name, n) in zip(axes, configs):
    ka, kb, ms, pb, Hb, inl_b, dt = find_box(name, n)
    outline = apply_h(Hb, np.float32([[0, 0], [box.shape[1], 0], [box.shape[1], box.shape[0]], [0, box.shape[0]], [0, 0]]))
    ax.imshow(scene, cmap="gray"); ax.plot(outline[:, 0], outline[:, 1], "lime", lw=2)
    ax.plot(pb[inl_b, 0], pb[inl_b, 1], "r.", ms=3)
    ax.set_title(f"{name}, не больше {n} точек: inliers {inl_b.sum()}, {dt * 1000:.0f} мс", fontsize=10); ax.axis("off")
    print(f"{name:4s} не больше {n:4d}: точек {len(ka):4d} и {len(kb):4d}, пар после Лоу {len(ms):3d}, inliers {inl_b.sum():3d}, время {dt * 1000:5.1f} мс")
plt.show()

**Что видно:** при одинаковом пределе в 500 точек SIFT находит коробку надёжнее: 59 пар после теста Лоу и 43 inliers против 30 и 26 у ORB. На маленькой коробке ORB даже не набирает 500 точек, а его двоичные дескрипторы реже проходят тест Лоу. Но ORB в четыре раза быстрее: 11 мс против 44 мс. Даже с 2000 точек он укладывается в 20 мс, вдвое быстрее SIFT, и находит 74 пары и 58 inliers, больше, чем SIFT с 500. Красные inliers лежат только на коробке: точки на соседних предметах не согласуются с её плоскостью. Ответ на вопрос, кто лучше, зависит от того, что фиксировать: число точек или время.

In [ ]:
ka, da = sift.detectAndCompute(box, None)
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, frac in zip(axes, [0.0, 0.3, 0.5, 0.7]):
    occ = scene.copy()
    x0, y0 = 100, 110
    occ[y0:y0 + int(180 * frac), x0:x0 + 220] = 128
    kb, db = sift.detectAndCompute(occ, None)
    ms = [m for m, n in cv.BFMatcher(cv.NORM_L2).knnMatch(da, db, k=2) if m.distance < 0.8 * n.distance]
    pa = np.float32([ka[m.queryIdx].pt for m in ms]); pb = np.float32([kb[m.trainIdx].pt for m in ms])
    Hb, mb = cv.findHomography(pa, pb, cv.RANSAC, 4.0) if len(ms) >= 4 else (None, None)
    print(f"закрыто {frac:.0%} области: пар {len(ms):3d}, inliers {0 if mb is None else int(mb.sum()):3d}")
    ax.imshow(occ, cmap="gray"); ax.axis("off")
    if Hb is not None:
        o = apply_h(Hb, np.float32([[0, 0], [box.shape[1], 0], [box.shape[1], box.shape[0]], [0, box.shape[0]], [0, 0]]))
        ax.plot(o[:, 0], o[:, 1], "lime", lw=2)
        ax.plot(pb[mb.ravel() > 0, 0], pb[mb.ravel() > 0, 1], "r.", ms=3)
    ax.set_title(f"закрыто {frac:.0%}: inliers {0 if mb is None else int(mb.sum())}", fontsize=10)
plt.show()

**Что видно:** если закрыть серым прямоугольником часть коробки, пар и inliers становится меньше: 79 inliers без перекрытия и 32 при закрытых 70% высоты. Но даже 32 согласованных пары на оставшейся части хватает, чтобы найти всю коробку. Локальные признаки поэтому хорошо переносят частичное перекрытие, в отличие от сравнения картинки целиком. На картинках зелёная рамка это коробка, перенесённая найденной гомографией, красные точки inliers: даже при закрытых 70% рамка ложится на коробку целиком, а inliers сидят только на открытой части.

## Когда гомографии мало

Гомография точна только для плоскости или для чистого поворота камеры. Сымитируем две плоскости на разной глубине: точки на дальней стене и на ближнем предмете, камера сдвинулась вбок. Подгоняем одну гомографию и смотрим ошибку на каждой плоскости.

In [ ]:
Kc = np.array([[700, 0, 400], [0, 700, 300], [0, 0, 1.0]])
far = np.c_[rng.uniform(-2, 2, (150, 2)), np.full(150, 8.0)]
near = np.c_[rng.uniform(-0.4, 0.4, (60, 2)), np.full(60, 2.0)]
X = np.vstack([far, near])
proj = lambda X, tx: (Kc @ (X - [tx, 0, 0]).T).T[:, :2] / X[:, 2:]
a_, b_ = proj(X, 0.0), proj(X, 0.3)
Hm, m_ = cv.findHomography(a_, b_, cv.RANSAC, 2.0)
err = np.linalg.norm(apply_h(Hm, a_) - b_, axis=1)
print("одна гомография на две плоскости: ошибка на дальней %.2f px, на ближней %.1f px; inliers: дальние %d из 150, ближние %d из 60"
      % (err[:150].mean(), err[150:].mean(), m_[:150].sum(), m_[150:].sum()))
Hm_near, mn_ = cv.findHomography(a_[150:], b_[150:], cv.RANSAC, 2.0)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, Hx, t in [(axes[0], Hm, "одна гомография по всем точкам"), (axes[1], Hm_near, "гомография ближней плоскости")]:
    pr = apply_h(Hx, a_)
    for sl, c, lab in [(slice(0, 150, 5), "C0", "дальняя стена"), (slice(150, None, 3), "C1", "ближний предмет")]:
        ax.plot(b_[sl, 0], b_[sl, 1], "o", mfc="none", color=c, ms=7, label=f"{lab}: где точки на кадре 2")
        ax.plot(pr[sl, 0], pr[sl, 1], "x", color=c, ms=6, label=f"{lab}: куда их переносит H")
        for p, q in zip(pr[sl], b_[sl]):
            ax.plot([p[0], q[0]], [p[1], q[1]], "-", color="0.4", lw=0.6)
    e = np.linalg.norm(pr - b_, axis=1)
    ax.set_title(f"{t}: ошибка дальних {e[:150].mean():.1f} px, ближних {e[150:].mean():.1f} px", fontsize=10)
    ax.set_xlim(100, 600); ax.set_ylim(500, 100); ax.set_aspect("equal")
axes[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(1.1, -0.08), ncol=2)
plt.show()

**Что видно:** камера сдвинулась на 30 см. RANSAC выбрал дальнюю плоскость, где больше точек: на ней ошибка ноль, а все 60 точек ближней плоскости стали выбросами с ошибкой 79 пикселей. Одна гомография описывает только одну плоскость. Для сцены с глубиной нужна фундаментальная матрица и триангуляция, это лекция 13. Для панорамы из этого следует правило: камеру надо вращать вокруг её оптического центра, а не переносить. На картинке показана каждая пятая точка стены и каждая третья точка предмета. Кружки показывают, где точки на самом деле видны на втором кадре, крестики показывают, куда их переносит гомография, чёрные отрезки это ошибка. Слева гомография дальней стены: синие крестики сидят в своих кружках, а оранжевые точки ближнего предмета промахиваются на десятки пикселей. Справа гомография, подогнанная по ближнему предмету: теперь ошибаются дальние точки. Какую плоскость ни выбери, вторая не совпадает.

*Попробуйте сами: сфотографируйте лист бумаги с текстом под углом, найдите его углы руками или по контуру и выпрямите гомографией. Потом снимите книжную полку и попробуйте склеить два кадра: где гомография ломается и почему?*